#### Signal Collection & Explicit Ratings

#### Sparsity Formula
$$\text{Sparsity(Density)} = \left(\frac{N_{\text{ratings}}}{N_{\text{users}} \times N_{\text{movies}}} \right) \times 100$$

Sparsity is the percentage of "empty" or "latent" cells in the matrix. It represents the information gap the Collaborative Filtering (CF) model must overcome.
$$\text{Sparsity} = \left( 1 - \frac{N_{\text{ratings}}}{N_{\text{users}} \times N_{\text{movies}}} \right) \times 100$$

- However, when the matrix is extremely sparse (as is typical in recommendation datasets like MovieLens), the number of actual ratings is very small compared to total possible entries.

- The difference between the two formulas is negligible (on the order of 0.0001–0.001%).

In [ ]:
# Load and inspect ratings data for CF

import pandas as pd
import numpy as np
from scipy.sparse import csr_matrix
import pickle
from pathlib import Path

print("=" * 40)
print("COLLABORATIVE FILTERING - DATA PREPARATION")
print("=" * 40)

# Load clean ratings
ratings = pd.read_csv("../data/processed/ratings_clean.csv")
print(f"\nRating shape: {ratings.shape}")
print(f"Columns: {ratings.columns.tolist()}")


# Basic Statistic
n_users = ratings["userId"].nunique()
n_movies = ratings["tmdbId"].nunique()
n_ratings = len(ratings)

print(f"\nUsers: {n_users:,}")
print(f"\nMovies: {n_movies:,}")
print(f"\nRatings: {n_ratings:,}")
# Sparsity = percentage of missing ratings
# Approximation valid for highly sparse matrices (difference negligible)
# Strict version: sparsity = (1 - (n_ratings / (n_users * n_movies))) * 100
print(f"Sparsity: {n_ratings / (n_users * n_movies) * 100:.4f}%")

SECTION 9: COLLABORATIVE FILTERING - DATA PREPARATION

Rating shape: (25981438, 6)
Columns: ['userId', 'movieId', 'rating', 'timestamp', 'tmdbId', 'year']

Users: 270,882

Movies: 44,702

Ratings: 25,981,438
Sparsity: 0.2146%


- The true sparsity is 99.7854% (meaning 99.7854% of possible user-movie pairs have no rating).
- The approximation 0.2146% is the proportion of filled entries — it is the inverse of the missing proportion.

##### Matrix Interaction & Sparsity Analysis
1. Theoretical Maximum Interactions (Total Search Space):
    - Formula: $N_{\text{users}} \times N_{\text{movies}}$
    - Calculation: $270,888 \times 44,702 \approx 12,109,235,376$ (12.11 Billion)
2. Observed Interactions (Actual Ratings):
    - Metric: $N_{\text{ratings}}$
    - Total: $25,981,438$ (~26 Million)
3. Matrix Density & Sparsity Results:
    - Density (Fill Rate):
    - $$\frac{25,981,438}{12,109,235,376} \times 100 \approx \mathbf{0.2146\%}$$
4. Sparsity (Missing Data):
    - $$100\% - 0.2146\% \approx \mathbf{99.7854\%}$$

#### Interpretation:
With a density of 0.2146%, the dataset exceeds the minimum threshold (0.1%) required for a stable Collaborative Filtering model. The system possesses sufficient interaction "connective tissue" to proceed with SVD++ Matrix Factorization.


### User Mean-Centering (also known as Mean-Normalization)

#### The Formula
$$r_{ui}^* = r_{ui} - \bar{r}_u$$

Where:

- $r_{ui}^*$: The Mean-Centered Rating (the result).
- $r_{ui}$: The Original Rating given by user $u$ to movie $i$ (e.g., 4 stars).
- $\bar{r}_u$: The Average Rating of all movies rated by user $u$.

In [ ]:
# User mean-centering - VECTORIZED (remove optimist/pessimist bias)

print("\n" + "=" * 60)
print("USER MEAN-CENTERING (VECTORIZED)")
print("=" * 60)

# Compute user means (fast, groupby returns small DataFrame)
user_means = ratings.groupby('userId')['rating'].mean().rename('user_mean')

# Merge means back (vectorized operation)
ratings = ratings.merge(user_means, left_on='userId', right_index=True)

# Vectorized centering (NO apply, NO lambda)
ratings['rating_centered'] = ratings['rating'] - ratings['user_mean']

# Drop the temporary column
ratings = ratings.drop('user_mean', axis=1)

print(f"Centered ratings created for {len(ratings):,} rows")
print("\nCentered rating statistics:")
print(ratings['rating_centered'].describe())


STEP 9.2: USER MEAN-CENTERING (VECTORIZED)
Centered ratings created for 25,981,438 rows

Centered rating statistics:
count    2.598144e+07
mean    -3.199727e-18
std      9.529301e-01
min     -4.490079e+00
25%     -5.333333e-01
50%      9.375000e-02
75%      6.400990e-01
max      4.285714e+00
Name: rating_centered, dtype: float64


In [17]:
# Double-centering - VECTORIZED

print("\n" + "=" * 60)
print("DOUBLE-CENTERING (VECTORIZED)")
print("=" * 60)

# Compute item means of centered ratings
item_means = ratings.groupby('tmdbId')['rating_centered'].mean().rename('item_mean')

# Merge and vectorized operation
ratings = ratings.merge(item_means, left_on='tmdbId', right_index=True)
ratings['rating_double_centered'] = ratings['rating_centered'] - ratings['item_mean']
ratings = ratings.drop('item_mean', axis=1)

print("Double-centered ratings statistics:")
print(ratings['rating_double_centered'].describe())


DOUBLE-CENTERING (VECTORIZED)


Double-centered ratings statistics:
count    2.598144e+07
mean    -1.801583e-17
std      8.722091e-01
min     -4.981469e+00
25%     -4.806821e-01
50%      6.404592e-02
75%      5.663130e-01
max      5.248576e+00
Name: rating_double_centered, dtype: float64


In [18]:
# IMPLICIT FEEDBACK SIGNALS

print("\n" + "=" * 60)
print("IMPLICIT FEEDBACK SIGNALS")
print("=" * 60)

# Binary implicit: 1 if positive interaction (rating >= 3.5)
ratings['implicit_binary'] = (ratings['rating'] >= 3.5).astype(int)

# Confidence-weighted implicit: scale to 0.1–1.0
ratings['implicit_confidence'] = ratings['rating'] / 5.0

print("Implicit signals added.")
print(ratings[['userId', 'tmdbId', 'rating', 'rating_double_centered', 
               'implicit_binary', 'implicit_confidence']].head(10))

# Save centered ratings with implicit signals
ratings.to_csv('../data/processed/ratings_centered.csv', index=False)
print("\n✓ Saved: ratings_centered.csv")


IMPLICIT FEEDBACK SIGNALS
Implicit signals added.
   userId  tmdbId  rating  rating_double_centered  implicit_binary  \
0   38150    1600     4.0               -0.103286                1   
1   44717    8012     3.0               -0.461146                0   
2   44717     807     5.0                1.115038                1   
3   44717     623     3.0               -0.701052                0   
4   45491    8844     4.0                0.549650                1   
5  126622     807     5.0                0.389602                1   
6  126622     629     5.0                0.201711                1   
7  126622   97406     4.0                0.095431                1   
8  126622   11010     5.0                0.448255                1   
9   45491   11860     5.0                1.408991                1   

   implicit_confidence  
0                  0.8  
1                  0.6  
2                  1.0  
3                  0.6  
4                  0.8  
5                  1.0  
6  

In [19]:
# CREATE INTERACTIONS_CLEAN.CSV

print("\n" + "=" * 60)
print("CREATE INTERACTIONS_CLEAN.CSV")
print("=" * 60)

# Create interactions DataFrame with ALL signal types
interactions = pd.DataFrame({
    'userId': ratings['userId'],
    'tmdb_id': ratings['tmdbId'],
    'explicit_value': ratings['rating_double_centered'],
    'implicit_binary': ratings['implicit_binary'],
    'implicit_confidence': ratings['implicit_confidence'],
    'type': 'explicit'  # All are explicit ratings with implicit derived
})

interactions.to_csv('../data/processed/interactions_clean.csv', index=False)
print(f"✓ Saved: interactions_clean.csv ({len(interactions):,} interactions)")
print(f"Columns: {interactions.columns.tolist()}")


CREATE INTERACTIONS_CLEAN.CSV
✓ Saved: interactions_clean.csv (25,981,438 interactions)
Columns: ['userId', 'tmdb_id', 'explicit_value', 'implicit_binary', 'implicit_confidence', 'type']


In [20]:
# BUILD SPARSE MATRIX & MAPPERS

print("\n" + "=" * 60)
print("SECTION 10.2-10.3: BUILD SPARSE MATRIX & MAPPERS")
print("=" * 60)

# Load your centered ratings (with implicit signals)
ratings = pd.read_csv('../data/processed/ratings_centered.csv')
print(f"Ratings loaded: {len(ratings):,}")

# CREATE USER MAPPERS
print("\n" + "-" * 40)
print("Creating user mappers...")

unique_users = sorted(ratings['userId'].unique())
user_mapper = {uid: idx for idx, uid in enumerate(unique_users)}
user_inv_mapper = {idx: uid for uid, idx in user_mapper.items()}
print(f"Users mapped: {len(user_mapper):,}")

# LOAD CBF MOVIE MAPPERS (for consistency)
print("\nLoading CBF movie mappers...")

with open("../models/cbf/movie_id_to_idx.pkl", "rb") as f:
    movie_id_to_idx = pickle.load(f)

with open("../models/cbf/idx_to_movie_id.pkl", "rb") as f:
    idx_to_movie_id = pickle.load(f)

print(f"Movies from CBF: {len(movie_id_to_idx):,}")

# MAP RATINGS TO INDICES
print("\n" + "-" * 40)
print("Mapping ratings to indices...")

ratings['user_idx'] = ratings['userId'].map(user_mapper)
ratings['movie_idx'] = ratings['tmdbId'].map(movie_id_to_idx)

# Verify no missing mappings
missing_users = ratings['user_idx'].isna().sum()
missing_movies = ratings['movie_idx'].isna().sum()

print(f"Missing user mappings: {missing_users:,}")
print(f"Missing movie mappings: {missing_movies:,}")

# Filter out movies not in CBF index
if missing_movies > 0:
    print(f"\nFiltering out {missing_movies} ratings with missing movie mappings...")
    ratings = ratings.dropna(subset=['movie_idx']).copy()
    print(f"Ratings after filtering: {len(ratings):,}")

# BUILD SPARSE MATRIX (using double-centered ratings)
print("\n" + "-" * 40)
print("Building sparse matrix...")

R = csr_matrix(
    (
        ratings['rating_double_centered'].astype(np.float32),
        (ratings['user_idx'].astype(int), ratings['movie_idx'].astype(int))
    ),
    shape=(len(user_mapper), len(movie_id_to_idx)),
    dtype=np.float32
)

print(f"Sparse matrix shape: {R.shape}")
print(f"Non-zero entries: {R.nnz:,}")
print(f"Density: {R.nnz / (R.shape[0] * R.shape[1]) * 100:.6f}%")

# SAVE ALL CF ARTIFACTS
print("\n" + "-" * 40)
print("Saving CF artifacts...")

# Save mappers
with open('../models/cf/user_mapper.pkl', 'wb') as f:
    pickle.dump(user_mapper, f)
print("✓ Saved: user_mapper.pkl")

with open('../models/cf/user_inv_mapper.pkl', 'wb') as f:
    pickle.dump(user_inv_mapper, f)
print("✓ Saved: user_inv_mapper.pkl")

with open('../models/cf/movie_mapper.pkl', 'wb') as f:
    pickle.dump(movie_id_to_idx, f)
print("✓ Saved: movie_mapper.pkl")

with open('../models/cf/movie_inv_mapper.pkl', 'wb') as f:
    pickle.dump(idx_to_movie_id, f)
print("✓ Saved: movie_inv_mapper.pkl")

# Save matrix
save_npz("../data/processed/interactions_matrix.npz", R)
print("✓ Saved: interactions_matrix.npz")

# Save Surprise-ready data
surprise_data = ratings[['userId', 'tmdbId', 'rating_double_centered']].copy()
surprise_data.columns = ['userId', 'tmdbId', 'rating']
surprise_data.to_csv('../data/processed/ratings_for_surprise.csv', index=False)
print("✓ Saved: ratings_for_surprise.csv")


SECTION 10.2-10.3: BUILD SPARSE MATRIX & MAPPERS
Ratings loaded: 25,981,438

----------------------------------------
Creating user mappers...
Users mapped: 270,882

Loading CBF movie mappers...
Movies from CBF: 45,423

----------------------------------------
Mapping ratings to indices...
Missing user mappings: 0
Missing movie mappings: 1

Filtering out 1 ratings with missing movie mappings...
Ratings after filtering: 25,981,437

----------------------------------------
Building sparse matrix...
Sparse matrix shape: (270882, 45423)
Non-zero entries: 25,981,437
Density: 0.211158%

----------------------------------------
Saving CF artifacts...
✓ Saved: user_mapper.pkl
✓ Saved: user_inv_mapper.pkl
✓ Saved: movie_mapper.pkl
✓ Saved: movie_inv_mapper.pkl
✓ Saved: interactions_matrix.npz
✓ Saved: ratings_for_surprise.csv


In [21]:
# FINAL VERIFICATION

print("\n" + "=" * 60)
print("FINAL CF DATA VERIFICATION")
print("=" * 60)

print(f"Users in matrix: {R.shape[0]:,}")
print(f"Movies in matrix: {R.shape[1]:,}")
print(f"Total interactions: {R.nnz:,}")
print(f"Matrix sparsity: {100 - (R.nnz / (R.shape[0] * R.shape[1]) * 100):.4f}% empty")

# Sample check
sample_user = ratings['userId'].iloc[0]
sample_user_idx = user_mapper[sample_user]
sample_movie = ratings['tmdbId'].iloc[0]
sample_movie_idx = movie_id_to_idx[sample_movie]
sample_value = R[sample_user_idx, sample_movie_idx]

print(f"\nSample verification:")
print(f"  User {sample_user} → index {sample_user_idx}")
print(f"  Movie {sample_movie} → index {sample_movie_idx}")
print(f"  Matrix value: {sample_value:.3f}")
print(f"  Original double-centered: {ratings['rating_double_centered'].iloc[0]:.3f}")


FINAL CF DATA VERIFICATION
Users in matrix: 270,882
Movies in matrix: 45,423
Total interactions: 25,981,437
Matrix sparsity: 99.7888% empty

Sample verification:
  User 38150 → index 38149
  Movie 1600 → index 1136
  Matrix value: -0.103
  Original double-centered: -0.103
